# Linear-Elastic Plate with Hole — Provenance Plots

This notebook fetches benchmark provenance data from RoHub and visualises how the simulation results depend on the mesh element size and the polynomial degree of the isoparametric elements.

Each series in the plots corresponds to one combination of simulation tool and element degree. The x-axis uses a logarithmic scale; error plots additionally use a logarithmic y-axis so that convergence rates appear as straight lines.

## Setup

Import the plotting dependencies and the `semantic_benchmark` package used to query RoHub. The plotting helpers are defined here so the complete data-to-figure workflow is visible in the notebook.

In [ ]:
!pip install "semantic-benchmark<=0.3.2"

In [5]:
import colorsys
import hashlib
from collections import defaultdict
from typing import Any, Sequence

import matplotlib.pyplot as plt
import pandas as pd
from semantic_benchmark.rohub import load_benchmark_metric_data

In [6]:
_CATEGORICAL_PALETTE = [
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100",
    "#e87ba4", "#008300", "#4a3aa7", "#e34948",
]
_LINESTYLES = ["-", "--", "-.", ":"]
_MARKERS = ["o", "s", "^", "D", "v", "P"]


def _stable_hash(label: str) -> int:
    """Return a deterministic hash so series styling is stable across runs."""
    digest = hashlib.md5(label.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big")


def _split_group_label(group: str) -> tuple[str, str]:
    """Split a 'Primary, extra=value' label into its two parts."""
    primary, _, remainder = group.partition(", ")
    return primary, remainder


def _assign_colors(labels: Sequence[str]) -> dict[str, str]:
    """Assign stable, non-colliding palette colors to up to eight labels."""
    assigned = {}
    slots_in_use = set()

    for label in sorted(labels):
        label_hash = _stable_hash(label)
        if len(assigned) < len(_CATEGORICAL_PALETTE):
            for probe in range(len(_CATEGORICAL_PALETTE)):
                slot = (label_hash + probe) % len(_CATEGORICAL_PALETTE)
                if slot not in slots_in_use:
                    slots_in_use.add(slot)
                    assigned[label] = _CATEGORICAL_PALETTE[slot]
                    break
        else:
            hue = (label_hash % 360) / 360.0
            red, green, blue = colorsys.hls_to_rgb(hue, 0.5, 0.65)
            assigned[label] = f"#{int(red * 255):02x}{int(green * 255):02x}{int(blue * 255):02x}"

    return assigned


def select_plot_columns(
    data: pd.DataFrame,
    parameters: Sequence[str],
    metrics: Sequence[str],
    group_column: str = "tool_name",
) -> pd.DataFrame:
    """Select plot columns and fold secondary parameters into the series label."""
    if not parameters:
        raise ValueError("At least one parameter is required for the x-axis.")
    if not metrics:
        raise ValueError("At least one metric is required for the y-axis.")

    extra_parameters = list(parameters[1:])
    required_columns = [group_column, parameters[0], metrics[0], *extra_parameters]
    missing_columns = [column for column in required_columns if column not in data.columns]
    if missing_columns:
        raise ValueError(
            "Cannot plot because these columns are missing: " + ", ".join(missing_columns)
        )

    plot_data = data.loc[:, required_columns].copy()
    if extra_parameters:
        plot_data[group_column] = plot_data.apply(
            lambda row: ", ".join(
                [str(row[group_column])]
                + [f"{parameter}={row[parameter]}" for parameter in extra_parameters]
            ),
            axis=1,
        )

    return plot_data.loc[:, [group_column, parameters[0], metrics[0]]].reset_index(drop=True)


def plot_provenance_graph(
    data: Sequence[Sequence[Any]],
    x_axis_label: str,
    y_axis_label: str,
    title: str,
    group_index: int = 0,
    x_axis_index: int = 1,
    y_axis_index: int = 2,
    output_file: str | None = None,
    figsize: tuple[int, int] = (12, 5),
    log_y: bool = False,
) -> None:
    """Plot grouped metric series from tabular benchmark results."""
    grouped_values = defaultdict(list)
    x_ticks = set()
    for row in data:
        group = str(row[group_index])
        x_value = float(row[x_axis_index])
        grouped_values[group].append((x_value, float(row[y_axis_index])))
        x_ticks.add(x_value)

    plt.figure(figsize=figsize)
    primary_labels = {_split_group_label(group)[0] for group in grouped_values}
    colors = _assign_colors(primary_labels)

    for group in sorted(grouped_values):
        x_values, y_values = zip(*sorted(grouped_values[group]))
        primary, remainder = _split_group_label(group)
        style_hash = _stable_hash(remainder)
        plt.plot(
            x_values,
            y_values,
            marker=_MARKERS[style_hash % len(_MARKERS)] if remainder else "o",
            linestyle=_LINESTYLES[style_hash % len(_LINESTYLES)] if remainder else "-",
            color=colors[primary],
            label=group,
        )

    if grouped_values:
        plt.legend()
    plt.xlabel(x_axis_label)
    plt.ylabel(y_axis_label)
    plt.title(title)
    plt.grid(True)
    plt.xscale("log")
    if log_y:
        plt.yscale("log")
    plt.xticks(sorted(x_ticks), [str(value) for value in sorted(x_ticks)], rotation=45)
    plt.tight_layout()

    if output_file:
        plt.savefig(output_file)
    else:
        plt.show()

## Configuration

In [7]:
BENCHMARK_NAME = "linear-elastic-plate-with-hole"
CODE_REPOSITORY_URL = "https://github.com/Simulation-Benchmarks/linear-elastic-plate-with-hole/tree/main"
USE_PRODUCTION_ROHUB = True  # set to False to use the development RoHub instance

## Fetch simulation results

`load_benchmark_metric_data` (from the `semantic_benchmark` package) connects to RoHub, finds every RO-Crate annotated with this benchmark and code repository URL, and queries each one for the requested parameters and metrics. The result is a pandas DataFrame where each row is one simulation run.

In [ ]:
data = load_benchmark_metric_data(
    benchmark_name=BENCHMARK_NAME,
    code_repository_url=CODE_REPOSITORY_URL,
    use_production_rohub=USE_PRODUCTION_ROHUB,
)

data

## Maximum von Mises Stress

Plot the maximum von Mises stress against the element size. A finer mesh generally resolves stress concentrations more accurately, so the value is expected to increase and converge towards the analytical solution as the element size decreases.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["element_size", "isoparametric_element_degree"],
    metrics=["max_von_mises_stress"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Element Size",
    y_axis_label="Max von Mises Stress",
    title="Max von Mises Stress vs Element Size",
)

## Maximum Displacement Error

Plot the maximum displacement error against the element size on a log–log scale. For a well-converging method the error decreases as a power of the element size, appearing as a straight line whose slope equals the convergence rate.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["element_size", "isoparametric_element_degree"],
    metrics=["max_displacement_error"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Element Size",
    y_axis_label="Max Displacement Error",
    title="Max Displacement Error vs Element Size",
    log_y=True,
)

## L2 Displacement Error

Plot the L2 norm of the displacement error against the element size on a log–log scale. The L2 error averages the error over the whole domain.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["element_size", "isoparametric_element_degree"],
    metrics=["l2_error_displacement"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Element Size",
    y_axis_label="L2 Error Displacement",
    title="L2 Error Displacement vs Element Size",
    log_y=True,
)